In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
path = r"C:\Users\13926\Atlantic-Project\AmesHousingClean.csv"
df = pd.read_csv(path)

In [3]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [4]:
# Encode 'Neighborhood' by the average sale price in each neighborhood
neighborhood_avg = X_train.join(y_train).groupby('Neighborhood')['SalePrice'].mean()
X_train['Neighborhood_encoded'] = X_train['Neighborhood'].map(neighborhood_avg)
X_test['Neighborhood_encoded'] = X_test['Neighborhood'].map(neighborhood_avg)
X_test['Neighborhood_encoded'] = X_test['Neighborhood_encoded'].fillna(neighborhood_avg.mean())
X_train.drop(columns='Neighborhood', inplace=True)
X_test.drop(columns='Neighborhood', inplace=True)

In [5]:
# Scale using Standard Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# Fit the data to Linear Regression and see the result
lin_regressor = LinearRegression()
lin_regressor.fit(X_train_scaled, y_train)
y_train_pred_lin = lin_regressor.predict(X_train_scaled)
y_pred_lin = lin_regressor.predict(X_test_scaled)
print(f'MAE: {mean_absolute_error(y_test, y_pred_lin)}')
print(f'RMSE: {root_mean_squared_error(y_test, y_pred_lin)}')
print(f'R-Squared score (training): {r2_score(y_train, y_train_pred_lin)}')
print(f'R-Squared score (testing): {r2_score(y_test, y_pred_lin)}')

MAE: 20581.828485488866
RMSE: 31312.644514102023
R-Squared score (training): 0.8240222911908476
R-Squared score (testing): 0.8346907517336511


In [ ]:
# Fit with KNN Regressor
knn_regressor = KNeighborsRegressor(n_neighbors=5)
knn_regressor.fit(X_train_scaled, y_train)
y_train_pred_knn = knn_regressor.predict(X_train_scaled)
y_pred_knn = knn_regressor.predict(X_test_scaled)
print(f'MAE: {mean_absolute_error(y_test, y_pred_knn)}')
print(f'R-Squared score (training): {r2_score(y_train, y_train_pred_knn)}')
print(f'R-Squared score (testing): {r2_score(y_test, y_pred_knn)}')

MAE: 18314.852962962967
R-Squared score (training): 0.9132909928512098
R-Squared score (testing): 0.862417102975511


In [ ]:
# Fine-tune KNN Regressor with grid search
param_grid_knn = {'n_neighbors': range(1, 31, 2),
                  'weights': ['uniform', 'distance'],
                  'metric': ['euclidean', 'manhattan', 'minkowski'], 
                  'n_jobs': [None, -1]}
grid_knn = GridSearchCV(KNeighborsRegressor(), param_grid_knn, cv=5, scoring='neg_mean_absolute_error')
grid_knn.fit(X_train_scaled, y_train)
print(grid_knn.best_params_)

{'metric': 'manhattan', 'n_jobs': None, 'n_neighbors': 13, 'weights': 'distance'}


In [ ]:
# Fit the fine-tuned KNN Regressor
knn_regressor = KNeighborsRegressor(**grid_knn.best_params_)
knn_regressor.fit(X_train_scaled, y_train)
y_train_pred_knn = knn_regressor.predict(X_train_scaled)
y_pred_knn = knn_regressor.predict(X_test_scaled)
print(f'MAE: {mean_absolute_error(y_test, y_pred_knn)}')
print(f'R-Squared score (training): {r2_score(y_train, y_train_pred_knn)}')
print(f'R-Squared score (testing): {r2_score(y_test, y_pred_knn)}')

MAE: 17091.28460530381
R-Squared score (training): 0.9999923534303838
R-Squared score (testing): 0.8724444058987917


In [ ]:
# Try with Random Forest Regressor
rf_regressor = RandomForestRegressor()
rf_regressor.fit(X_train_scaled, y_train)
y_train_pred_rf = rf_regressor.predict(X_train_scaled)
y_pred_rf = rf_regressor.predict(X_test_scaled)
print(f'MAE: {mean_absolute_error(y_test, y_pred_rf)}')
print(f'R-Squared score (training): {r2_score(y_train, y_train_pred_rf)}')
print(f'R-Squared score (testing): {r2_score(y_test, y_pred_rf)}')

MAE: 16588.07595229277
R-Squared score (training): 0.9844792088206327
R-Squared score (testing): 0.8904647905746454


In [ ]:
# Also do grid search on Random Forest Regressor
param_grid = {
    'n_estimators': [100, 200],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [10, 20, 50, None],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(RandomForestRegressor(), param_grid=param_grid, cv=5)
grid_search.fit(X_train_scaled, y_train)
print('Best Parameters', grid_search.best_params_)


Best Parameters {'max_depth': 50, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 200}
Best Estimator {'max_depth': 50, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 200}


In [ ]:
# Fit the fine-tuned model
rf_regressor = RandomForestRegressor(**grid_search.best_params_)
rf_regressor.fit(X_train_scaled, y_train)
y_train_pred_rf = rf_regressor.predict(X_train_scaled)
y_pred_rf = rf_regressor.predict(X_test_scaled)
print(f'MAE: {mean_absolute_error(y_test, y_pred_rf)}')
print(f'MAPE: {mean_absolute_percentage_error(y_test, y_pred_rf)}')
print(f'R-Squared score (training): {r2_score(y_train, y_train_pred_rf)}')
print(f'R-Squared score (testing): {r2_score(y_test, y_pred_rf)}')

MAE: 15972.47621313933
MAPE: 0.09282871418834875
R-Squared score (training): 0.9869192032213011
R-Squared score (testing): 0.8936238574321004
